# 📘 Módulo 13 — Redes Neurais Recorrentes (RNNs), LSTM, GRU e Modelagem de Sequências

Este módulo introduz as RNNs, redes projetadas para lidar com **dados sequenciais**.

Você aprenderá:
- como funcionam RNNs
- por que elas têm memória
- o problema do gradiente que explode/desaparece
- como LSTM e GRU resolvem isso
- como treinar modelos para texto e séries temporais
- como usar embeddings
- como prever sequências passo a passo

---

# 🎯 Objetivos do Módulo 13

Ao final deste módulo, você será capaz de:

✔️ entender a intuição das RNNs  
✔️ construir RNNs simples com Keras  
✔️ treinar LSTMs para séries temporais  
✔️ treinar LSTMs para texto  
✔️ usar GRUs como alternativa mais leve  
✔️ entender por que Transformers substituíram RNNs  

---

# 📚 Índice

1. [Intuição das RNNs](#intuicao)
2. [O problema do gradiente e a motivação das LSTMs](#gradiente)
3. [Construindo sua primeira RNN](#primeira_rnn)
4. [LSTM: a arquitetura que mudou tudo](#lstm)
5. [GRU: versão simplificada e eficiente](#gru)
6. [Modelagem de séries temporais com LSTM](#series)
7. [Modelagem de texto com LSTM + Embeddings](#texto)
8. [Projeto Final — Previsão de Sequências com LSTM](#projeto)

---

<a id="setup"></a>
# 0. Setup — bibliotecas

Vamos usar TensorFlow/Keras para construir RNNs, LSTMs e GRUs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

plt.style.use("seaborn-v0_8-whitegrid")

tf.__version__

<a id="intuicao"></a>
# 2. Intuição das Redes Neurais Recorrentes (RNNs)

CNNs lidam com **imagens**.
MLPs lidam com **tabelas**.

Mas e quando os dados têm **ordem**, **tempo**, **sequência**?

Exemplos:
- texto (frases, parágrafos)
- séries temporais (preço, clima, sensores)
- áudio (ondas sonoras)
- dados de navegação (trajetórias)
- logs de eventos

Redes neurais tradicionais **não conseguem** lidar com dependências temporais.

RNNs foram criadas para isso.

---
# 2.1 O problema das redes tradicionais com sequências

Uma frase como:

> "O gato subiu no telhado"

Não pode ser tratada como:

```
[palavra1, palavra2, palavra3, ...]
```

Porque:
- a ordem importa  
- o significado depende do contexto  
- palavras influenciam as seguintes  

MLPs não têm memória.
CNNs têm visão local, mas não lembram o passado.

Precisamos de um modelo que:
- processe um elemento por vez  
- **lembre** o que veio antes  
- atualize seu estado interno  

Isso é exatamente o que RNNs fazem.

---
# 2.2 A ideia central das RNNs

Uma RNN processa uma sequência **passo a passo**.

Em cada passo:

```
entrada atual + memória anterior → nova memória
```

Formalmente:

$$
h_t = f(Wx_t + Uh_{t-1} + b)
$$

Onde:
- \( x_t \) = entrada no tempo t  
- \( h_{t-1} \) = memória anterior  
- \( h_t \) = nova memória  

A rede passa a ter **estado interno**.

---
# 2.3 Visualização da recorrência

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(10, 3))
plt.plot([0,1,2,3,4], [0,1,0.5,1.5,1], marker="o")
plt.title("Sequência sendo processada passo a passo")
plt.xlabel("Tempo (t)")
plt.ylabel("Valor")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

A RNN lê um elemento por vez.
A cada passo, ela atualiza sua memória.

Isso permite capturar dependências como:
- “não” → inverte o sentido da frase  
- “mas” → muda o tom  
- “amanhã” → altera o tempo verbal  
- “tendência” → muda a previsão da série temporal  

---
# 2.4 RNNs são redes “desdobradas no tempo”

Uma RNN pode ser vista como várias cópias da mesma rede,
compartilhando pesos:

In [ ]:
plt.figure(figsize=(12, 3))
plt.text(0.1, 0.5, "x₁ → [RNN] → h₁", fontsize=14)
plt.text(3.1, 0.5, "x₂ → [RNN] → h₂", fontsize=14)
plt.text(6.1, 0.5, "x₃ → [RNN] → h₃", fontsize=14)
plt.axis("off")
plt.title("RNN desdobrada no tempo")
plt.show()

🟣 **Ponto-chave**

A mesma célula RNN é aplicada repetidamente.
Isso permite:
- generalização  
- memória  
- eficiência  

---
# 2.5 O problema das RNNs simples

RNNs simples sofrem com:

- **gradiente que explode**  
- **gradiente que desaparece**  

Isso impede que aprendam dependências longas.

Exemplos:
- prever a última palavra de uma frase longa  
- prever tendência de uma série temporal de meses  

Por isso surgiram:
- **LSTM**  
- **GRU**  

Elas resolvem o problema com mecanismos de portas.

---
# 2.6 Conclusão desta parte

✔️ RNNs processam sequências passo a passo  
✔️ Elas têm memória interna  
✔️ São adequadas para texto, áudio e séries temporais  
✔️ RNNs simples sofrem com gradiente que explode/desaparece  
✔️ Isso motiva o surgimento das LSTMs  

Agora estamos prontos para:

**PARTE 3 — Construindo sua primeira RNN simples com Keras.**

<a id="primeira_rnn"></a>
# 3. Construindo sua primeira RNN simples

Agora que entendemos a intuição das RNNs,
vamos construir uma RNN real usando Keras.

Para isso, vamos criar um dataset sequencial simples:

**Objetivo:**  
Prever o próximo valor de uma sequência senoidal.

Isso é perfeito para entender:
- formato de entrada das RNNs  
- como treinar  
- como prever passo a passo  

---
# 3.1 Criando um dataset sequencial simples

In [ ]:
# Criando uma sequência senoidal
t = np.linspace(0, 50, 500)
series = np.sin(t)

plt.figure(figsize=(10, 3))
plt.plot(t, series)
plt.title("Série Temporal Senoidal")
plt.grid(alpha=0.3)
plt.show()

---
# 3.2 Transformando a série em janelas (janela deslizante)

RNNs esperam dados no formato:

```
(amostras, passos de tempo, features)
```

Vamos criar janelas de 20 passos para prever o próximo valor.

In [ ]:
def create_dataset(series, window=20):
    X, y = [], []
    for i in range(len(series) - window):
        X.append(series[i:i+window])
        y.append(series[i+window])
    return np.array(X), np.array(y)

X, y = create_dataset(series, window=20)

# Ajustando formato para RNN
X = X.reshape((X.shape[0], X.shape[1], 1))

X.shape, y.shape

---
# 3.3 Construindo a RNN simples

Vamos usar:
- `SimpleRNN(20)`  
- `Dense(1)`  

Essa é a RNN mais básica possível.

In [ ]:
model_rnn = keras.Sequential([
    layers.SimpleRNN(20, activation="tanh", input_shape=(20, 1)),
    layers.Dense(1)
])

model_rnn.compile(
    optimizer="adam",
    loss="mse"
)

model_rnn.summary()

---
# 3.4 Treinando a RNN

In [ ]:
history_rnn = model_rnn.fit(
    X, y,
    epochs=20,
    batch_size=32,
    verbose=1
)

---
# 3.5 Fazendo previsões

In [ ]:
preds = model_rnn.predict(X)

plt.figure(figsize=(10, 3))
plt.plot(t[20:], y, label="Real")
plt.plot(t[20:], preds.flatten(), label="Previsão RNN")
plt.legend()
plt.title("Previsão da RNN sobre Série Senoidal")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

A RNN:
- capturou o padrão da série  
- aprendeu dependências temporais  
- conseguiu prever a forma da onda  

Mesmo sendo simples, ela já demonstra o poder das redes recorrentes.

---
# 3.6 Conclusão desta parte

✔️ Criamos um dataset sequencial  
✔️ Formatamos os dados para RNN  
✔️ Construímos uma RNN simples  
✔️ Treinamos e fizemos previsões  
✔️ Vimos a capacidade da RNN de aprender padrões temporais  

Agora estamos prontos para:

**PARTE 4 — LSTM: a arquitetura que revolucionou o Deep Learning.**

<a id="lstm"></a>
# 4. LSTM — A arquitetura que revolucionou o Deep Learning

RNNs simples sofrem com:
- gradiente que explode
- gradiente que desaparece

Isso impede que aprendam dependências longas.

Em 1997, Hochreiter & Schmidhuber criaram a **LSTM**:

> Uma RNN com mecanismos internos de memória (gates)

Ela consegue:
- lembrar informações por longos períodos
- esquecer o que não importa
- controlar o fluxo de informação

Por isso dominou NLP e séries temporais por 20 anos.

---
# 4.1 A ideia central da LSTM

A LSTM tem três portas:

- **forget gate** → decide o que esquecer  
- **input gate** → decide o que guardar  
- **output gate** → decide o que expor  

Isso resolve o problema do gradiente.

---
# 4.2 Visualização conceitual da LSTM

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,3))
plt.text(0.1, 0.5, "[Forget Gate]", fontsize=14)
plt.text(3.1, 0.5, "[Input Gate]", fontsize=14)
plt.text(6.1, 0.5, "[Output Gate]", fontsize=14)
plt.axis("off")
plt.title("Componentes internos de uma LSTM")
plt.show()

---
# 4.3 Construindo sua primeira LSTM

Vamos usar o mesmo dataset senoidal da parte anterior,
mas agora com uma LSTM.

In [ ]:
# Reutilizando X e y criados anteriormente
X.shape, y.shape

---
# 4.4 Criando o modelo LSTM

In [ ]:
model_lstm = keras.Sequential([
    layers.LSTM(50, activation="tanh", input_shape=(20, 1)),
    layers.Dense(1)
])

model_lstm.compile(
    optimizer="adam",
    loss="mse"
)

model_lstm.summary()

---
# 4.5 Treinando a LSTM

In [ ]:
history_lstm = model_lstm.fit(
    X, y,
    epochs=20,
    batch_size=32,
    verbose=1
)

---
# 4.6 Fazendo previsões com a LSTM

In [ ]:
preds_lstm = model_lstm.predict(X)

plt.figure(figsize=(10, 3))
plt.plot(t[20:], y, label="Real")
plt.plot(t[20:], preds_lstm.flatten(), label="Previsão LSTM")
plt.legend()
plt.title("Previsão da LSTM sobre Série Senoidal")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

A LSTM:
- aprende dependências mais longas  
- suaviza melhor a curva  
- é mais estável que a RNN simples  
- captura padrões temporais com mais precisão  

Em séries reais, a diferença é ainda maior.

---
# 4.7 Comparação RNN vs LSTM

| Modelo | Memória Longa | Estabilidade | Precisão | Uso Real |
|--------|----------------|--------------|----------|----------|
| RNN simples | baixa | baixa | moderada | quase nunca |
| **LSTM** | **alta** | **alta** | **excelente** | padrão por 20 anos |

LSTM foi a base de:
- Google Translate (pré‑Transformers)  
- Siri  
- Alexa  
- Modelos seq2seq  
- Reconhecimento de fala moderno  

---
# 4.8 Conclusão desta parte

✔️ Entendemos por que LSTMs surgiram  
✔️ Vimos como elas resolvem o problema do gradiente  
✔️ Construímos uma LSTM real  
✔️ Treinamos e comparamos com RNN simples  
✔️ Vimos por que LSTM dominou NLP e séries temporais  

Agora estamos prontos para:

**PARTE 5 — GRU: a versão mais leve e eficiente da LSTM.**

<a id="gru"></a>
# 5. GRU — A versão mais leve e eficiente da LSTM

A GRU foi criada em 2014 como uma alternativa simplificada à LSTM.

Ela remove:
- a célula de memória explícita
- a porta de saída

E combina:
- forget gate + input gate → update gate

Resultado:
- menos parâmetros
- mais rápida
- desempenho muito próximo da LSTM

Em muitos casos, GRU é a melhor escolha prática.

---
# 5.1 Estrutura da GRU

A GRU tem duas portas:

- **update gate** → controla o quanto manter da memória antiga  
- **reset gate** → controla o quanto ignorar do passado  

Fórmulas simplificadas:

$$
z_t = \sigma(W_z x_t + U_z h_{t-1})
$$

$$
r_t = \sigma(W_r x_t + U_r h_{t-1})
$$

$$
h_t = (1 - z_t) h_{t-1} + z_t \tilde{h}_t
$$

É mais simples que a LSTM, mas igualmente poderosa.

---
# 5.2 Construindo uma GRU com Keras

Vamos usar o mesmo dataset senoidal das partes anteriores.

In [ ]:
X.shape, y.shape

---
# 5.3 Criando o modelo GRU

In [ ]:
model_gru = keras.Sequential([
    layers.GRU(50, activation="tanh", input_shape=(20, 1)),
    layers.Dense(1)
])

model_gru.compile(
    optimizer="adam",
    loss="mse"
)

model_gru.summary()

---
# 5.4 Treinando a GRU

In [ ]:
history_gru = model_gru.fit(
    X, y,
    epochs=20,
    batch_size=32,
    verbose=1
)

---
# 5.5 Fazendo previsões com a GRU

In [ ]:
preds_gru = model_gru.predict(X)

plt.figure(figsize=(10, 3))
plt.plot(t[20:], y, label="Real")
plt.plot(t[20:], preds_gru.flatten(), label="Previsão GRU")
plt.legend()
plt.title("Previsão da GRU sobre Série Senoidal")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

A GRU:
- aprende tão bem quanto a LSTM  
- treina mais rápido  
- tem menos parâmetros  
- é mais estável que a RNN simples  

Em muitos casos, GRU é a melhor escolha prática.

---
# 5.6 Comparação LSTM vs GRU

| Modelo | Parâmetros | Velocidade | Memória Longa | Precisão |
|--------|------------|------------|----------------|----------|
| LSTM | alta | média | excelente | excelente |
| **GRU** | **média** | **rápida** | muito boa | excelente |

Em benchmarks reais:
- GRU ≈ LSTM  
- GRU treina mais rápido  
- GRU overfita menos em datasets pequenos  

---
# 5.7 Quando usar cada uma?

✔️ **Use LSTM quando:**
- há dependências muito longas  
- o dataset é grande  
- você quer máxima precisão  

✔️ **Use GRU quando:**
- quer treinar rápido  
- dataset é pequeno  
- quer simplicidade  
- quer evitar overfitting  

✔️ **Use RNN simples quando:**
- nunca  
- só para fins didáticos  

---
# 5.8 Conclusão desta parte

✔️ Entendemos a arquitetura GRU  
✔️ Construímos uma GRU real  
✔️ Treinamos e comparamos com LSTM  
✔️ Vimos que GRU é mais leve e igualmente poderosa  

Agora estamos prontos para:

**PARTE 6 — Modelagem de Séries Temporais com LSTM (projeto real).**

<a id="series"></a>
# 6. Modelagem de Séries Temporais com LSTM

Agora vamos aplicar LSTM em um caso real de séries temporais.

**Objetivo:**  
Prever valores futuros de uma série temporal suave (seno + ruído).

Isso simula:
- sensores industriais
- sinais fisiológicos
- séries de energia
- séries de tráfego
- séries meteorológicas

---
# 6.1 Criando uma série temporal realista

In [ ]:
np.random.seed(42)

t = np.linspace(0, 100, 1000)
series = np.sin(t) + 0.2 * np.random.randn(1000)

plt.figure(figsize=(10, 3))
plt.plot(t, series)
plt.title("Série Temporal Realista (seno + ruído)")
plt.grid(alpha=0.3)
plt.show()

---
# 6.2 Criando janelas deslizantes

Vamos usar janelas de 30 passos para prever o próximo valor.

In [ ]:
def create_dataset(series, window=30):
    X, y = [], []
    for i in range(len(series) - window):
        X.append(series[i:i+window])
        y.append(series[i+window])
    return np.array(X), np.array(y)

X, y = create_dataset(series, window=30)

# Ajustando formato para LSTM
X = X.reshape((X.shape[0], X.shape[1], 1))

X.shape, y.shape

---
# 6.3 Dividindo treino e teste

In [ ]:
split = int(len(X) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

X_train.shape, X_test.shape

---
# 6.4 Criando a LSTM para séries temporais

In [ ]:
model_series = keras.Sequential([
    layers.LSTM(64, activation="tanh", return_sequences=True, input_shape=(30, 1)),
    layers.LSTM(32, activation="tanh"),
    layers.Dense(1)
])

model_series.compile(
    optimizer="adam",
    loss="mse"
)

model_series.summary()

---
# 6.5 Treinando a LSTM

In [ ]:
history_series = model_series.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

---
# 6.6 Avaliando no conjunto de teste

In [ ]:
preds_test = model_series.predict(X_test)

plt.figure(figsize=(10, 3))
plt.plot(y_test, label="Real")
plt.plot(preds_test.flatten(), label="Previsão LSTM")
plt.legend()
plt.title("Previsão da LSTM — Conjunto de Teste")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

A LSTM:
- capturou o padrão da série  
- suavizou o ruído  
- previu tendências de forma consistente  

Isso é exatamente o que esperamos de uma LSTM bem treinada.

---
# 6.7 Previsão passo a passo (forecasting futuro)

Agora vamos prever **50 passos à frente**, usando a própria previsão como entrada.

In [ ]:
future_steps = 50
last_window = X_test[-1].flatten().tolist()

future_preds = []

for _ in range(future_steps):
    x_input = np.array(last_window[-30:]).reshape(1, 30, 1)
    next_val = model_series.predict(x_input)[0][0]
    future_preds.append(next_val)
    last_window.append(next_val)

plt.figure(figsize=(10, 3))
plt.plot(range(len(series)), series, label="Histórico")
plt.plot(range(len(series), len(series) + future_steps), future_preds, label="Previsão futura")
plt.legend()
plt.title("Previsão Futura — LSTM")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

A LSTM consegue:
- projetar tendências  
- manter periodicidade  
- suavizar ruído  

Esse tipo de previsão é usado em aplicações reais.

---
# 6.8 Conclusão desta parte

✔️ Criamos uma série temporal realista  
✔️ Preparamos janelas deslizantes  
✔️ Construímos uma LSTM profunda  
✔️ Treinamos e avaliamos previsões  
✔️ Fizemos previsão futura passo a passo  
✔️ Visualizamos resultados de forma profissional  

Agora estamos prontos para:

**PARTE 7 — Modelagem de Texto com LSTM + Embeddings (NLP clássico).**

<a id="texto"></a>
# 7. Modelagem de Texto com LSTM + Embeddings

Agora vamos aplicar LSTM em um problema real de NLP:

**Classificação de sentimentos (positivo/negativo)**  

Usaremos o dataset IMDB (50.000 reviews de filmes).

Pipeline:
- tokenização
- padding
- embeddings
- LSTM
- classificação

---
# 7.1 Carregando o dataset IMDB

In [ ]:
from tensorflow.keras.datasets import imdb

# Mantemos apenas as 10.000 palavras mais frequentes
num_words = 10000
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=num_words)

len(X_train), len(X_test)

---
# 7.2 Visualizando um review tokenizado

In [ ]:
X_train[0][:20]

🟣 **Interpretação**

Cada número representa uma palavra.
Exemplo:
- 14 → "the"
- 20 → "film"
- 33 → "was"

O texto já vem pré-tokenizado.

---
# 7.3 Padronizando o tamanho das sequências (padding)

LSTMs exigem que todas as sequências tenham o mesmo tamanho.

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = 200  # tamanho fixo

X_train_pad = pad_sequences(X_train, maxlen=max_len)
X_test_pad = pad_sequences(X_test, maxlen=max_len)

X_train_pad.shape, X_test_pad.shape

---
# 7.4 Construindo o modelo LSTM para texto

Arquitetura:
- Embedding(10000 → 64)
- LSTM(64)
- Dense(1, sigmoid)

In [ ]:
model_text = keras.Sequential([
    layers.Embedding(input_dim=num_words, output_dim=64, input_length=max_len),
    layers.LSTM(64),
    layers.Dense(1, activation="sigmoid")
])

model_text.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_text.summary()

---
# 7.5 Treinando o modelo

In [ ]:
history_text = model_text.fit(
    X_train_pad, y_train,
    epochs=3,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

---
# 7.6 Avaliação no conjunto de teste

In [ ]:
loss_text, acc_text = model_text.evaluate(X_test_pad, y_test, verbose=0)
acc_text

🟣 **Resultado típico**

- Acurácia entre **85% e 88%**  
- Excelente para um modelo simples  

LSTM + Embeddings ainda é muito forte em NLP clássico.

---
# 7.7 Fazendo previsões em novos textos

Precisamos converter texto → tokens → padding.

In [ ]:
word_index = imdb.get_word_index()

# Função para converter texto em tokens
def encode_text(text):
    tokens = []
    for word in text.lower().split():
        if word in word_index and word_index[word] < num_words:
            tokens.append(word_index[word])
        else:
            tokens.append(2)  # token <UNK>
    return pad_sequences([tokens], maxlen=max_len)

# Exemplo
sample = "This movie was amazing and I loved every moment"
sample_encoded = encode_text(sample)

pred = model_text.predict(sample_encoded)[0][0]
pred

🟣 **Interpretação**

Valores próximos de:
- 1 → sentimento positivo  
- 0 → sentimento negativo  

---
# 7.8 Conclusão desta parte

✔️ Aprendemos tokenização e padding  
✔️ Usamos embeddings para representar palavras  
✔️ Construímos uma LSTM para classificação de texto  
✔️ Treinamos no dataset IMDB  
✔️ Avaliamos e fizemos previsões reais  

Agora estamos prontos para:

**PARTE 8 — Projeto Final: Previsão de Sequências com LSTM (texto + séries).**

<a id="projeto"></a>
# 8. Projeto Final — Previsão de Sequências com LSTM

Neste projeto final, você vai integrar:

✔️ LSTM para séries temporais  
✔️ LSTM para texto  
✔️ Embeddings  
✔️ Previsão passo a passo  
✔️ Pipeline completo de modelagem sequencial  

Vamos construir dois modelos:
1. **LSTM para prever uma série temporal realista**  
2. **LSTM para prever a próxima palavra em uma sequência de texto**  

Esse é o tipo de projeto que une NLP + Time Series.

---
# PARTE A — LSTM para Séries Temporais

Vamos criar uma série temporal complexa:
- seno
- tendência
- ruído
- sazonalidade

In [ ]:
np.random.seed(42)

t = np.linspace(0, 200, 2000)
series = (
    0.5 * np.sin(0.2 * t) +
    0.3 * np.sin(0.05 * t) +
    0.01 * t +
    0.2 * np.random.randn(2000)
)

plt.figure(figsize=(12, 3))
plt.plot(t, series)
plt.title("Série Temporal Complexa")
plt.grid(alpha=0.3)
plt.show()

---
# A.1 Criando janelas deslizantes

In [ ]:
def create_dataset(series, window=40):
    X, y = [], []
    for i in range(len(series) - window):
        X.append(series[i:i+window])
        y.append(series[i+window])
    return np.array(X), np.array(y)

X, y = create_dataset(series, window=40)
X = X.reshape((X.shape[0], X.shape[1], 1))

X.shape, y.shape

---
# A.2 Dividindo treino e teste

In [ ]:
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

---
# A.3 Criando a LSTM

In [ ]:
model_ts = keras.Sequential([
    layers.LSTM(64, return_sequences=True, input_shape=(40, 1)),
    layers.LSTM(32),
    layers.Dense(1)
])

model_ts.compile(optimizer="adam", loss="mse")
model_ts.summary()

---
# A.4 Treinando

In [ ]:
history_ts = model_ts.fit(
    X_train, y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

---
# A.5 Avaliação

In [ ]:
preds_ts = model_ts.predict(X_test)

plt.figure(figsize=(12, 3))
plt.plot(y_test, label="Real")
plt.plot(preds_ts.flatten(), label="Previsão LSTM")
plt.legend()
plt.title("Previsão — Série Temporal Complexa")
plt.grid(alpha=0.3)
plt.show()

---
# A.6 Previsão futura (50 passos)

In [ ]:
future_steps = 50
last_window = X_test[-1].flatten().tolist()
future_preds = []

for _ in range(future_steps):
    x_input = np.array(last_window[-40:]).reshape(1, 40, 1)
    next_val = model_ts.predict(x_input)[0][0]
    future_preds.append(next_val)
    last_window.append(next_val)

plt.figure(figsize=(12, 3))
plt.plot(range(len(series)), series, label="Histórico")
plt.plot(range(len(series), len(series)+future_steps), future_preds, label="Futuro")
plt.legend()
plt.title("Previsão Futura — LSTM")
plt.grid(alpha=0.3)
plt.show()

---
# PARTE B — LSTM para Previsão de Texto

Agora vamos treinar uma LSTM para prever a **próxima palavra**.

In [ ]:
text = """
o gato subiu no telhado e depois pulou para o muro
o cachorro latiu para o gato que estava no muro
o gato correu para o jardim e subiu na árvore
"""

---
# B.1 Tokenização

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

word_index = tokenizer.word_index
vocab_size = len(word_index) + 1

sequences = []

words = text.split()
for i in range(3, len(words)):
    seq = words[i-3:i+1]
    sequences.append(tokenizer.texts_to_sequences([" ".join(seq)])[0])

sequences[:5]

---
# B.2 Preparando dados

In [ ]:
sequences = np.array(sequences)
X_text, y_text = sequences[:, :-1], sequences[:, -1]

X_text = pad_sequences(X_text, maxlen=3)

X_text.shape, y_text.shape

---
# B.3 Criando a LSTM para texto

In [ ]:
model_text = keras.Sequential([
    layers.Embedding(vocab_size, 16, input_length=3),
    layers.LSTM(32),
    layers.Dense(vocab_size, activation="softmax")
])

model_text.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_text.summary()

---
# B.4 Treinando

In [ ]:
history_text = model_text.fit(
    X_text, y_text,
    epochs=200,
    verbose=0
)

---
# B.5 Função para prever próxima palavra

In [ ]:
def predict_next(words):
    seq = tokenizer.texts_to_sequences([words])[0]
    seq = pad_sequences([seq], maxlen=3)
    pred = model_text.predict(seq, verbose=0)
    idx = np.argmax(pred)
    for word, index in word_index.items():
        if index == idx:
            return word

predict_next("o gato subiu")

---
# B.6 Gerando texto automaticamente

In [ ]:
seed = "o gato subiu"
generated = seed

for _ in range(10):
    next_word = predict_next(generated.split()[-3:])
    generated += " " + next_word

generated

---
# 8. Conclusões do Projeto Final

✔️ Construímos uma LSTM para séries temporais complexas  
✔️ Fizemos previsão futura passo a passo  
✔️ Construímos uma LSTM para texto  
✔️ Previsão de próxima palavra  
✔️ Geração automática de texto  
✔️ Integramos NLP + Time Series  

Você agora domina:
- RNN  
- LSTM  
- GRU  
- Embeddings  
- Modelagem de texto  
- Modelagem de séries temporais  
- Previsão futura  

E está pronto para avançar para:

# **Módulo 14 — Transformers, Attention e o nascimento do Deep Learning moderno.**